# Gameweek planner

Expected points for **every** player in the game, plus your own squad analysed.

Set your team in section 1, then run top to bottom. Everything except sections
4–6 works without a team set.

## 1. Settings - edit these

In [1]:
# ─── YOUR TEAM ────────────────────────────────────────────────────────────
# Leave as None to read FPL_TEAM_ID from .env (set it once and forget it).
# Set a number here to look at a different team just for this run.
#
# To find an id: view the team on the FPL site and read it out of the URL -
#     fantasy.premierleague.com/entry/ 1234567 /event/4
#                                      ^^^^^^^
TEAM_ID = None

# Or list your 15 by name instead (used only if TEAM_ID is None):
MY_TEAM = []              # e.g. ["Haaland", "Saka", "Guéhi", ...]

# ─── MINUTES ──────────────────────────────────────────────────────────────
# Playing time is modelled automatically (availability flags, recent starts,
# rotation risk, and the rule that each team starts exactly 11). Where you know
# better than the model, override it per player in
# config/minutes_overrides.yaml and re-run:
#
#     players:
#       Haaland: 90     # pin to a full match
#       Saka: 0         # rule out
#
# Section 10 lists the players most worth a manual call.

# ─── FIXTURE OUTLOOK (section 7) ──────────────────────────────────────────
LOOKAHEAD = 10            # how many gameweeks to chart
WINDOW    = 5             # how many to rank teams over
FREE_TRANSFERS = 1        # how many transfers you have without taking a hit
DISCOUNT  = 0.85          # weight on gameweek k = DISCOUNT ** (k-1)
                          #   1.0  = every gameweek in the window counts equally
                          #   0.85 = balanced (default)
                          #   0.7  = concentrate on the next two or three

# ─── OTHER ────────────────────────────────────────────────────────────────
GAMEWEEK = None           # None = next gameweek

# Every API call is cached (bootstrap/fixtures 1hr, live gameweek stats 1hr,
# odds 15min), so re-running this notebook normally costs nothing. Set this to
# True to force a genuine refresh of all of it - prices, injury news, live
# minutes, and market odds - right before you act on the output. It burns a
# small amount of Odds API quota (2 credits/fixture) each time, so leave it
# False for routine re-runs and flip it on only when it matters.
REFRESH = False

## 2. Run the model

In [2]:
# Pick up edits to the fplfh package without restarting the kernel. Without
# this, a running kernel keeps the version of a module it first imported, and
# any function added since fails to import until you restart.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass          # not running under IPython

import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "fplfh").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)

from fplfh.pipeline import run
from fplfh.fpl import FPLClient
from fplfh.optimise import best_xi, optimise_free_hit
from fplfh.model import COMPONENTS

client = FPLClient()
res = run(event=GAMEWEEK, max_age=0 if REFRESH else None)

print(f"\nGameweek {res.event}   |   "
      f"{100*res.exchange_share():.0f}% of expected points from market odds")
res.fixture_table()

Odds API: priced 10/10 fixtures  [quota unknown (no live call yet this session)]
GW5: 659 players, 10 fixtures

Gameweek 5   |   100% of expected points from market odds


,fixture,kickoff,xG_home,xG_away,P(home),P(draw),P(away),CS_home,CS_away,source
0,BRE v CHE,2026-09-18 19:00:00+00:00,1.59,1.73,0.345,0.251,0.404,0.177,0.203,oddsapi:betfair_ex_uk:h2h+totals(3)
1,TOT v AVL,2026-09-19 11:30:00+00:00,1.66,1.15,0.480,0.268,0.251,0.317,0.190,oddsapi:betfair_ex_uk:h2h+totals(5)
2,BHA v ARS,2026-09-19 14:00:00+00:00,0.99,1.84,0.194,0.241,0.565,0.159,0.370,oddsapi:betfair_ex_uk:h2h+totals(5)
3,EVE v IPS,2026-09-19 14:00:00+00:00,1.83,1.07,0.540,0.251,0.208,0.344,0.161,oddsapi:betfair_ex_uk:h2h+totals(5)
4,NEW v HUL,2026-09-19 14:00:00+00:00,2.02,1.04,0.597,0.217,0.186,0.353,0.133,oddsapi:betfair_ex_uk:h2h+totals(5)
5,NFO v COV,2026-09-19 16:30:00+00:00,1.91,0.95,0.590,0.236,0.174,0.386,0.148,oddsapi:betfair_ex_uk:h2h+totals(5)
6,BOU v LIV,2026-09-20 13:00:00+00:00,1.50,1.84,0.304,0.251,0.445,0.159,0.222,oddsapi:betfair_ex_uk:h2h+totals(4)
7,LEE v CRY,2026-09-20 13:00:00+00:00,1.79,1.04,0.542,0.248,0.210,0.353,0.166,oddsapi:betfair_ex_uk:h2h+totals(5)
8,MCI v SUN,2026-09-20 13:00:00+00:00,2.43,0.69,0.756,0.159,0.085,0.501,0.088,oddsapi:betfair_ex_uk:h2h+totals(5)
9,FUL v MUN,2026-09-20 15:30:00+00:00,1.35,1.80,0.277,0.254,0.469,0.165,0.259,oddsapi:betfair_ex_uk:h2h+totals(4)


## 3. Every player, ranked

The full table. `xp` is the sum of the ten component columns - reading the
components tells you *why* a player rates, which is usually more useful than the
total.

In [3]:
ALL = res.players.copy()
cols = ["web_name","team","position","price","opponent","is_home"] + COMPONENTS + ["xp"]

print(f"{len(ALL)} players projected for GW{res.event}\n")
ALL.head(40)[cols].round(2)

659 players projected for GW5



,web_name,team,position,price,opponent,is_home,xp_minutes,xp_goals,xp_assists,xp_clean_sheet,xp_conceded,xp_defcon,xp_saves,xp_cards,xp_bonus,xp_pens,xp
0,Haaland,MCI,FWD,15.5,SUN,H,1.78,4.20,0.47,0.00,0.00,0.01,0.0,-0.09,1.55,-0.08,7.83
1,Gibbs-White,NFO,MID,8.0,COV,H,1.94,2.30,1.26,0.38,0.00,0.02,0.0,-0.15,0.93,-0.07,6.62
2,Saka,ARS,MID,9.5,BHA,A,1.88,2.71,0.73,0.36,0.00,0.16,0.0,-0.14,0.96,-0.06,6.59
3,Barry,EVE,FWD,5.6,IPS,H,1.87,3.32,0.22,0.00,0.00,0.00,0.0,-0.11,1.21,-0.06,6.45
4,B.Fernandes,MUN,MID,12.0,FUL,A,1.91,2.29,1.08,0.26,0.00,0.05,0.0,-0.15,0.90,-0.06,6.27
5,Wissa,NEW,FWD,6.2,HUL,H,1.94,2.84,0.14,0.00,0.00,0.02,0.0,-0.16,1.03,0.00,5.81
6,Calvert-Lewin,LEE,FWD,6.0,CRY,H,1.89,2.78,0.17,0.00,0.00,0.00,0.0,-0.18,1.01,-0.06,5.61
7,Guéhi,MCI,DEF,6.0,SUN,H,1.83,1.15,0.40,1.83,-0.14,0.17,0.0,-0.14,0.49,0.00,5.61
8,Mbeumo,MUN,MID,7.9,FUL,A,1.89,2.21,0.59,0.26,0.00,0.04,0.0,-0.17,0.77,0.00,5.59
9,Isak,LIV,FWD,9.1,BOU,A,1.77,2.73,0.20,0.00,0.00,0.00,0.0,-0.10,0.99,0.00,5.59


In [4]:
# Filter it however you like - a few useful starting points.
def top(position=None, max_price=None, min_xp=0.0, team=None, n=20):
    d = ALL.copy()
    if position:  d = d[d.position == position]
    if max_price: d = d[d.price <= max_price]
    if team:      d = d[d.team == team]
    d = d[d.xp >= min_xp]
    return d.head(n)[["web_name","team","position","price","opponent","xmins","xp"]].round(2)

print("Best value under £6.0m:")
display(top(max_price=6.0, n=12))

Best value under £6.0m:


,web_name,team,position,price,opponent,xmins,xp
3,Barry,EVE,FWD,5.6,IPS,78.82,6.45
6,Calvert-Lewin,LEE,FWD,6.0,CRY,79.98,5.61
7,Guéhi,MCI,DEF,6.0,SUN,77.16,5.61
11,Bogle,LEE,DEF,4.5,CRY,79.89,5.29
13,Stach,LEE,MID,6.0,CRY,80.81,5.25
16,Hall,NEW,DEF,5.2,HUL,82.40,5.15
17,Murillo,NFO,DEF,5.5,COV,82.37,5.13
21,Ndoye,NFO,MID,5.5,COV,75.35,4.84
22,Barnes,NEW,MID,6.0,HUL,82.40,4.77
23,Muharemović,LEE,DEF,5.0,CRY,81.99,4.75


In [5]:
print("Top 8 in each position:\n")
for pos in ("GKP","DEF","MID","FWD"):
    print(f"── {pos} " + "─"*60)
    print(top(position=pos, n=8).to_string(index=False))
    print()

Top 8 in each position:

── GKP ────────────────────────────────────────────────────────────
  web_name team position  price opponent  xmins   xp
Donnarumma  MCI      GKP    5.5      SUN  76.14 3.96
      Sels  NFO      GKP    5.0      COV  82.36 3.80
  Horníček  NEW      GKP    5.0      HUL  82.40 3.63
  Trafford  LEE      GKP    5.0      CRY  81.52 3.61
  Pickford  EVE      GKP    5.5      IPS  81.23 3.56
      Raya  ARS      GKP    6.0      BHA  78.85 3.56
    Kinsky  TOT      GKP    4.5      AVL  77.14 3.25
   Lammens  MUN      GKP    4.9      FUL  75.49 2.93

── DEF ────────────────────────────────────────────────────────────
   web_name team position  price opponent  xmins   xp
      Guéhi  MCI      DEF    6.0      SUN  77.16 5.61
      Bogle  LEE      DEF    4.5      CRY  79.89 5.29
       Hall  NEW      DEF    5.2      HUL  82.40 5.15
    Murillo  NFO      DEF    5.5      COV  82.37 5.13
Muharemović  LEE      DEF    5.0      CRY  81.99 4.75
      Rúben  MCI      DEF    5.5     

In [6]:
# Points per million - for filling out the cheap end of a squad
v = ALL[ALL.xp > 1.0].copy()
v["xp_per_million"] = v.xp / v.price
print("Best value per million (xP > 1.0):")
v.nlargest(20, "xp_per_million")[
    ["web_name","team","position","price","opponent","xp","xp_per_million"]].round(2)

Best value per million (xP > 1.0):


,web_name,team,position,price,opponent,xp,xp_per_million
11,Bogle,LEE,DEF,4.5,CRY,5.29,1.18
3,Barry,EVE,FWD,5.6,IPS,6.45,1.15
27,Justin,LEE,DEF,4.5,CRY,4.50,1.00
16,Hall,NEW,DEF,5.2,HUL,5.15,0.99
23,Muharemović,LEE,DEF,5.0,CRY,4.75,0.95
5,Wissa,NEW,FWD,6.2,HUL,5.81,0.94
6,Calvert-Lewin,LEE,FWD,6.0,CRY,5.61,0.94
7,Guéhi,MCI,DEF,6.0,SUN,5.61,0.94
17,Murillo,NFO,DEF,5.5,COV,5.13,0.93
68,Thomas,COV,DEF,4.0,NFO,3.69,0.92


## 4. Your team

Pulled straight from the FPL API using `FPL_TEAM_ID` from `.env`, or `TEAM_ID`
if you set one above.

Note the API only makes picks public once a gameweek's deadline has passed, so
this shows your most recent **confirmed** squad - transfers you have made since
will not appear.

In [7]:
from fplfh.config import fpl_team_id

# An explicit TEAM_ID above wins; otherwise fall back to .env.
team_id = TEAM_ID if TEAM_ID else fpl_team_id()
squad_df, my_budget, source = None, None, None

if team_id:
    picks, picked_gw = client.latest_picks(int(team_id))
    ids = [p["element"] for p in picks["picks"]]
    squad_df = ALL[ALL.player_id.isin(ids)].copy()
    eh = picks["entry_history"]
    my_budget = (eh["value"] + eh["bank"]) / 10.0
    info = client.entry(int(team_id))
    source = f'{info.get("name","?")} ({info.get("player_first_name","")} '\
             f'{info.get("player_last_name","")}) - squad as at GW{picked_gw}'
    print(source)
    print(f"  squad value L{eh['value']/10:.1f}m + bank L{eh['bank']/10:.1f}m "
          f"= L{my_budget:.1f}m")
    print(f"  overall rank {info.get('summary_overall_rank'):,}" if info.get("summary_overall_rank") else "")
    missing = set(ids) - set(squad_df.player_id)
    if missing:
        print(f"  note: {len(missing)} player(s) have no fixture this gameweek (blank)")

elif MY_TEAM:
    # Accent-insensitive on both sides: nobody types "Ødegaard" or "Groß" with
    # the special letters, and those are atomic in Unicode so a naive accent
    # strip mangles them ("Hojlund" -> "h jlund"). normalise() transliterates.
    from fplfh.naming import normalise

    ALL["_key"] = ALL.web_name.map(normalise)
    # normalise() strips brackets, so "Palmer (CHE)" becomes "palmer che" -
    # the key has to be built the same way or the qualifier never matches.
    ALL["_key_team"] = ALL._key + " " + ALL.team.str.lower()

    # 19 surnames are shared by two or three players (three Wilsons, three
    # Phillips, two Palmers), so a bare surname is genuinely ambiguous. Qualify
    # those with a club: "Palmer (CHE)".
    counts = ALL._key.value_counts()
    picked, unmatched, ambiguous = [], [], []
    for raw in MY_TEAM:
        k = normalise(raw)
        exact = ALL[ALL._key_team == k]
        if len(exact) == 1:
            picked.append(exact.index[0]); continue
        hits = ALL[ALL._key == k]
        if len(hits) == 1:
            picked.append(hits.index[0])
        elif len(hits) > 1:
            ambiguous.append((raw, hits[["web_name", "team", "position", "xp"]]))
        else:
            unmatched.append(raw)

    squad_df = ALL.loc[picked].copy()

    for raw, hits in ambiguous:
        print(f"  AMBIGUOUS '{raw}' - {len(hits)} players share that name. "
              f'Use e.g. "{raw} ({hits.iloc[0].team})":')
        print(hits.to_string(index=False))
    for raw in unmatched:
        close = ALL[ALL._key.str.startswith(normalise(raw)[:4])].web_name.head(4).tolist()
        print(f"  NOT FOUND '{raw}'" + (f" - did you mean {close}?" if close else ""))

    ALL = ALL.drop(columns=["_key", "_key_team"])
    squad_df = squad_df.drop(columns=["_key", "_key_team"])
    my_budget = squad_df.price.sum()
    source = "manual list"
    print(f"matched {len(squad_df)} of {len(MY_TEAM)} names, "
          f"total value L{my_budget:.1f}m")

else:
    print("No team set. Either put FPL_TEAM_ID in .env, or fill in TEAM_ID or")
    print("MY_TEAM in section 1, to use sections 4-6.")
    print("Everything else in this notebook works without it.")

palhinha colada (Alex Giffin) - squad as at GW4
  squad value L100.3m + bank L1.3m = L101.6m
  overall rank 745,351


In [8]:
if squad_df is not None and len(squad_df):
    d = squad_df.sort_values(["position","xp"], ascending=[True,False])
    print(f"Your {len(d)} players, projected for GW{res.event}:\n")
    display(d[["web_name","team","position","price","opponent","is_home","xmins"]
              + COMPONENTS + ["xp"]].round(2))
    print(f"\nsquad total xP (all {len(d)}): {d.xp.sum():.2f}")

Your 15 players, projected for GW5:



,web_name,team,position,price,opponent,is_home,xmins,xp_minutes,xp_goals,xp_assists,xp_clean_sheet,xp_conceded,xp_defcon,xp_saves,xp_cards,xp_bonus,xp_pens,xp
33,Calafiori,ARS,DEF,5.8,BHA,A,79.89,1.89,0.60,0.49,1.43,-0.25,0.03,0.00,-0.15,0.34,0.00,4.38
153,Diop,IPS,DEF,4.0,EVE,A,75.75,1.81,0.03,0.17,0.61,-0.56,0.57,0.00,-0.14,0.11,0.00,2.59
160,O'Reilly,MCI,DEF,6.4,SUN,H,41.90,1.12,0.19,0.33,0.90,-0.07,0.00,0.00,-0.08,0.15,0.00,2.54
207,Pedro Porro,TOT,DEF,5.5,AVL,H,38.72,1.00,0.39,0.19,0.55,-0.15,0.03,0.00,-0.08,0.11,0.00,2.04
226,Shaw,MUN,DEF,4.4,FUL,A,40.76,1.06,0.04,0.24,0.48,-0.20,0.15,0.00,-0.10,0.06,0.00,1.73
0,Haaland,MCI,FWD,15.5,SUN,H,74.45,1.78,4.20,0.47,0.00,0.00,0.01,0.00,-0.09,1.55,-0.08,7.83
6,Calvert-Lewin,LEE,FWD,6.0,CRY,H,79.98,1.89,2.78,0.17,0.00,0.00,0.00,0.00,-0.18,1.01,-0.06,5.61
59,João Pedro,CHE,FWD,7.8,BRE,A,60.43,1.49,1.67,0.18,0.00,0.00,0.00,0.00,-0.10,0.61,0.00,3.85
80,Raya,ARS,GKP,6.0,BHA,A,78.85,1.87,0.00,0.00,1.41,-0.25,0.00,0.39,-0.06,0.20,0.00,3.56
148,Verbruggen,BHA,GKP,4.5,ARS,H,76.22,1.81,0.00,0.09,0.61,-0.57,0.00,0.60,-0.05,0.13,0.00,2.62



squad total xP (all 15): 53.76


### Your best XI

Which of your 15 to start, who to captain, and the bench order. This chooses
nothing about *which* players you own - only how to line them up.

In [9]:
if squad_df is not None and len(squad_df) >= 11:
    mine = best_xi(squad_df)
    print(mine.summary())
    print(f"\nprojected total: {mine.starting_xp:.2f} points (captain doubled)")
    bench_pts = squad_df[~squad_df.player_id.isin(
        mine.players[mine.players.is_starter].player_id)].xp.sum()
    print(f"left on the bench: {bench_pts:.2f}")

Formation 3-4-3   cost L99.1m   XI xP 54.69 (incl. captain)
Captain: Haaland   Vice: Calvert-Lewin

  GKP  Raya               ARS  L 6.0  BHA      xP  3.56
  DEF  Calafiori          ARS  L 5.8  BHA      xP  4.38
  DEF  Diop               IPS  L 4.0  EVE      xP  2.59
  DEF  O'Reilly           MCI  L 6.4  SUN      xP  2.54
  MID  Palmer             CHE  L 9.7  BRE      xP  5.40
  MID  Szoboszlai         LIV  L 7.0  BOU      xP  4.67
  MID  Groß               BHA  L 5.7  ARS      xP  3.47
  MID  Tzolis             ARS  L 6.4  BHA      xP  2.95
  FWD  Haaland            MCI  L15.5  SUN      xP  7.83 (C)
  FWD  Calvert-Lewin      LEE  L 6.0  CRY      xP  5.61
  FWD  João Pedro         CHE  L 7.8  BRE      xP  3.85

  Bench:
  GKP  Verbruggen         BHA  L 4.5  ARS      xP  2.62
  DEF  Pedro Porro        TOT  L 5.5  AVL      xP  2.04
  DEF  Shaw               MUN  L 4.4  FUL      xP  1.73
  MID  Hughes             CRY  L 4.4  LEE      xP  0.52

projected total: 54.69 points (captain double

In [10]:
# Captain choice - the armband is worth a whole extra return, so it is the
# single biggest decision of the week.
if squad_df is not None and len(squad_df):
    print("Captaincy options from your squad:\n")
    display(squad_df.nlargest(5, "xp")[
        ["web_name","team","position","opponent","is_home","xp_goals","xp_assists",
         "xmins","xp"]].round(2))
    print("Doubling the top pick is worth an extra "
          f"{squad_df.xp.max():.2f} points in expectation.")

Captaincy options from your squad:



,web_name,team,position,opponent,is_home,xp_goals,xp_assists,xmins,xp
0,Haaland,MCI,FWD,SUN,H,4.20,0.47,74.45,7.83
6,Calvert-Lewin,LEE,FWD,CRY,H,2.78,0.17,79.98,5.61
10,Palmer,CHE,MID,BRE,A,1.76,0.97,81.21,5.40
25,Szoboszlai,LIV,MID,BOU,A,1.19,0.85,75.87,4.67
33,Calafiori,ARS,DEF,BHA,A,0.60,0.49,79.89,4.38


Doubling the top pick is worth an extra 7.83 points in expectation.


## 5. Transfers

For each player you own, the best alternative in the same position you could
afford. Selling price is approximated as the current price - FPL's actual rule
gives back half of any profit, so real funds may be slightly lower.

In [11]:
if squad_df is not None and len(squad_df):
    bank = 0.0 if team_id is None else (client.latest_picks(int(team_id))[0]
                                       ["entry_history"]["bank"] / 10.0)
    owned = set(squad_df.player_id)
    rows = []
    for _, p in squad_df.iterrows():
        budget = p.price + bank
        cands = ALL[(ALL.position == p.position) & (~ALL.player_id.isin(owned))
                    & (ALL.price <= budget) & (ALL.xp > p.xp)]
        if not len(cands):
            continue
        b = cands.nlargest(1, "xp").iloc[0]
        rows.append({"out": p.web_name, "out_team": p.team, "out_xp": round(p.xp,2),
                     "in": b.web_name, "in_team": b.team, "in_xp": round(b.xp,2),
                     "gain": round(b.xp - p.xp,2),
                     "cost": round(b.price - p.price,1)})
    if rows:
        t = pd.DataFrame(rows).sort_values("gain", ascending=False)
        print(f"Best single upgrade per player (bank L{bank:.1f}m):\n")
        display(t)
        print("A transfer costs 4 points unless it is free - only the top few")
        print("rows are likely to be worth taking a hit for.")
    else:
        print("No affordable upgrades found - your squad is already strong "
              "for this gameweek.")

Best single upgrade per player (bank L1.3m):



,out,out_team,out_xp,in,in_team,in_xp,gain,cost
13,Hughes,CRY,0.52,Ndoye,NFO,4.84,4.31,1.1
11,Pedro Porro,TOT,2.04,Guéhi,MCI,5.61,3.57,0.5
12,Shaw,MUN,1.73,Bogle,LEE,5.29,3.56,0.1
10,O'Reilly,MCI,2.54,Guéhi,MCI,5.61,3.07,-0.4
9,Diop,IPS,2.59,Bogle,LEE,5.29,2.70,0.5
4,João Pedro,CHE,3.85,Barry,EVE,6.45,2.61,-2.2
7,Tzolis,ARS,2.95,Rogers,CHE,5.28,2.32,1.3
2,Szoboszlai,LIV,4.67,Gibbs-White,NFO,6.62,1.95,1.0
6,Groß,BHA,3.47,Stach,LEE,5.25,1.78,0.3
8,Verbruggen,BHA,2.62,Donnarumma,MCI,3.96,1.35,1.0


A transfer costs 4 points unless it is free - only the top few
rows are likely to be worth taking a hit for.


## 6. The unconstrained best squad

What the optimiser would pick if you could start from scratch with your budget.
Useful as a benchmark - the gap between this and your XI is what a Free Hit or
Wildcard could theoretically buy you.

In [12]:
# A partial or empty squad gives a budget far too small for 15 players, so fall
# back to the game's standard 100.0 rather than asking for the impossible.
budget = my_budget if (my_budget and my_budget >= 80) else 100.0
if my_budget and my_budget < 80:
    print(f"(matched squad is only worth L{my_budget:.1f}m, too little for a "
          f"full 15 - using the standard L100.0m instead)\n")

dream = optimise_free_hit(ALL, budget=budget)
if dream is None:
    print(f"No legal 15-player squad fits inside L{budget:.1f}m.")
else:
    print(f"Best possible squad for L{budget:.1f}m:\n")
    print(dream.summary())

if dream is not None and squad_df is not None and len(squad_df) >= 11:
    print(f"\n{'─'*64}")
    print(f"  your XI          {mine.starting_xp:6.2f}")
    print(f"  best possible    {dream.starting_xp:6.2f}")
    print(f"  gap              {dream.starting_xp - mine.starting_xp:6.2f} points")
    print("\n  That gap needs 15 transfers to close, so treat it as a ceiling,")
    print("  not a target.")

Best possible squad for L101.6m:

Formation 3-4-3   cost L101.5m   XI xP 72.68 (incl. captain)
Captain: Haaland   Vice: Gibbs-White

  GKP  Donnarumma         MCI  L 5.5  SUN      xP  3.96
  DEF  Guéhi              MCI  L 6.0  SUN      xP  5.61
  DEF  Bogle              LEE  L 4.5  CRY      xP  5.29
  DEF  Hall               NEW  L 5.2  HUL      xP  5.15
  MID  Gibbs-White        NFO  L 8.0  COV      xP  6.62
  MID  Saka               ARS  L 9.5  BHA      xP  6.59
  MID  B.Fernandes        MUN  L12.0  FUL      xP  6.27
  MID  Stach              LEE  L 6.0  CRY      xP  5.25
  FWD  Haaland            MCI  L15.5  SUN      xP  7.83 (C)
  FWD  Barry              EVE  L 5.6  IPS      xP  6.45
  FWD  Wissa              NEW  L 6.2  HUL      xP  5.81

  Bench:
  GKP  Bentley            COV  L 4.0  NFO      xP  0.31
  DEF  Justin             LEE  L 4.5  CRY      xP  4.50
  DEF  Thomas             COV  L 4.0  NFO      xP  3.69
  MID  Rudoni             COV  L 5.0  NFO      xP  4.13

────────────

## 7. Fixture outlook

Who has the best run of games, and over what horizon.

Difficulty here isn't FPL's 1-5 rating, which is a coarse integer set before
the season and never revised. It comes from the same scoreline model as
everything else, in the units that actually score points: expected goals for
(attackers), clean sheet probability (defenders), expected goals against
(keepers).

The odds themselves only reach three to four gameweeks ahead - on the live
feed that was 24 days, covering two gameweeks across an international break.
Beyond that the ratings model fills in, and every fixture is tagged with which
was used.

Distant gameweeks are weighted down in the ranking, but not because they're
less predictable: measured across a full season, the correlation between a
team's attacking rating and its actual goals is flat at ~0.28-0.31 whether you
look one gameweek ahead or ten, so team strength is broadly stable over that
range. The discount is really about actionability - you'll make transfers
before then, so a far-off fixture should sway today's decision less. That
makes it a planning preference, not a fitted constant.

In [13]:
from fplfh.outlook import build_outlook, fixture_grid, rank_fixtures, compare_windows

outlook = build_outlook(client, res.minutes, res.fixtures, res.event,
                        n_events=LOOKAHEAD, verbose=True)
n_mkt = int((outlook.source == "market").sum())
print()
print(f"{len(outlook)} team-fixtures over GW{res.event}-{res.event + LOOKAHEAD - 1}")
print(f"  {n_mkt} priced from the market, {len(outlook) - n_mkt} from the ratings model")
print()
print("The schedule (H = home, A = away):")
fixture_grid(outlook)

odds covered 20/100 fixtures in GW5-14

200 team-fixtures over GW5-14
  40 priced from the market, 160 from the ratings model

The schedule (H = home, A = away):


event,5,6,7,8,9,10,11,12,13,14
team,,,,,,,,,,
ARS,BHA (A),LEE (H),NFO (A),EVE (H),LIV (A),HUL (H),NEW (A),MCI (H),BRE (A),TOT (A)
AVL,TOT (A),BRE (H),NEW (A),MCI (H),FUL (H),MUN (A),SUN (H),IPS (A),EVE (H),CRY (H)
BHA,ARS (H),SUN (A),CRY (H),LIV (A),MCI (A),BRE (H),HUL (A),NEW (H),BOU (A),NFO (A)
BOU,LIV (H),CHE (A),SUN (H),MUN (A),LEE (H),IPS (A),NFO (H),FUL (A),BHA (H),HUL (H)
BRE,CHE (H),AVL (A),LIV (H),HUL (A),NFO (H),BHA (A),EVE (H),MUN (A),ARS (H),MCI (H)
CHE,BRE (A),BOU (H),EVE (A),TOT (H),MUN (H),SUN (A),LEE (H),NFO (A),CRY (H),LIV (H)
COV,NFO (A),NEW (H),TOT (A),FUL (H),SUN (H),EVE (A),CRY (H),LEE (A),IPS (H),MUN (A)
CRY,LEE (A),NFO (H),BHA (A),NEW (H),TOT (A),LIV (H),COV (A),HUL (H),CHE (A),AVL (A)
EVE,IPS (H),HUL (A),CHE (H),ARS (A),NEW (A),COV (H),BRE (A),LIV (H),AVL (A),FUL (H)


In [14]:
# The same grid as numbers - expected goals FOR each team, per gameweek.
# Blanks show as NaN; doubles are summed.
print("Expected goals for, by gameweek:")
fixture_grid(outlook, value="xg_for").round(2)

Expected goals for, by gameweek:


event,5,6,7,8,9,10,11,12,13,14
team,,,,,,,,,,
ARS,1.84,2.31,1.23,1.77,1.34,1.82,1.56,1.54,1.36,1.47
AVL,1.15,1.47,1.22,1.20,1.57,1.04,1.41,1.15,1.38,1.68
BHA,0.99,1.42,2.61,1.63,1.44,2.13,1.70,2.44,1.62,1.48
BOU,1.50,1.38,1.71,1.26,1.66,1.40,1.50,1.47,1.85,1.72
BRE,1.59,1.42,1.79,1.44,1.63,1.55,1.82,1.37,1.46,1.58
CHE,1.73,2.10,1.34,1.87,1.70,1.37,1.72,1.20,2.11,1.71
COV,0.95,1.26,1.23,1.68,1.51,1.14,1.80,1.13,1.60,1.11
CRY,1.04,1.24,1.33,1.78,1.30,1.54,1.37,1.61,1.28,1.18
EVE,1.83,1.39,1.64,0.96,1.36,1.76,1.19,1.52,1.17,1.76


### Best fixtures over your chosen window

In [15]:
for metric, who in (("attack", "forwards and attacking midfielders"),
                    ("defence", "defenders and goalkeepers")):
    r = rank_fixtures(outlook, window=WINDOW, discount=DISCOUNT, metric=metric)
    print(f"-- BEST for {metric.upper()} over the next {WINDOW} GWs ({who})")
    print(r.head(6)[["fixtures", "blanks", "doubles", "xg_for", "clean_sheet",
                     "home_games", "market_priced", "opponents"]].round(3).to_string())
    print(f"   avoid: {', '.join(r.tail(3).index)}")
    print()

-- BEST for ATTACK over the next 5 GWs (forwards and attacking midfielders)
      fixtures  blanks  doubles  xg_for  clean_sheet  home_games  market_priced                               opponents
team                                                                                                                   
MCI          5       0        0   1.965        0.337           3              2  SUN(H), LIV(A), IPS(H), AVL(A), BHA(H)
MUN          5       0        0   1.804        0.248           2              2  FUL(A), TOT(H), LEE(A), BOU(H), CHE(A)
CHE          5       0        0   1.758        0.231           3              2  BRE(A), BOU(H), EVE(A), TOT(H), MUN(H)
ARS          5       0        0   1.746        0.368           2              2  BHA(A), LEE(H), NFO(A), EVE(H), LIV(A)
NEW          5       0        0   1.627        0.282           3              2  HUL(H), COV(A), AVL(H), CRY(A), EVE(H)
LIV          5       0        0   1.598        0.205           3              2  BOU

In [16]:
# Does the discount change the answer? If a team's rank holds across discounts
# its run is uniformly good; if it swings, the good fixtures are clustered at
# one end of the window.
# Ranks each team at several discounts, to show how sensitive the order is.
# Always includes whatever DISCOUNT is set above, so the sort column exists
# whichever value you choose.
def discount_swing(metric, discounts=(1.0, 0.85, 0.7, 0.5)):
    ranks = {}
    for d in sorted(set(discounts) | {float(DISCOUNT)}, reverse=True):
        label = f"discount {d:g}" + ("  <-yours" if d == float(DISCOUNT) else "")
        ranks[label] = rank_fixtures(
            outlook, window=WINDOW, discount=d, metric=metric
        ).score.rank(ascending=False).astype(int)
    t = pd.DataFrame(ranks)
    t["swing"] = t.max(axis=1) - t.min(axis=1)
    return t.sort_values(next(c for c in t.columns if "<-yours" in c))


print("ATTACK - rank by discount. A large swing means the good fixtures are")
print("clustered early or late rather than spread evenly:")
display(discount_swing("attack").head(12))

ATTACK - rank by discount. A large swing means the good fixtures are
clustered early or late rather than spread evenly:


,discount 1,discount 0.85 <-yours,discount 0.7,discount 0.5,swing
team,,,,,
MCI,1,1,1,1,0
MUN,2,2,2,3,1
CHE,3,3,4,5,2
ARS,4,4,3,2,2
NEW,7,5,5,4,3
LIV,6,6,6,6,0
BHA,5,7,10,14,9
BRE,8,8,7,9,2
NFO,10,9,8,7,3


In [17]:
print("DEFENCE - rank by discount.")
print("Clean-sheet probability varies proportionally more across teams than")
print("expected goals does (coefficient of variation ~0.22 vs ~0.14), so WHICH")
print("fixture a defender has matters relatively more. The rank swing itself is")
print("similar for both, though - neither is reliably the more volatile.")
display(discount_swing("defence").head(12))

DEFENCE - rank by discount.
Clean-sheet probability varies proportionally more across teams than
expected goals does (coefficient of variation ~0.22 vs ~0.14), so WHICH
fixture a defender has matters relatively more. The rank swing itself is
similar for both, though - neither is reliably the more volatile.


,discount 1,discount 0.85 <-yours,discount 0.7,discount 0.5,swing
team,,,,,
ARS,1,1,1,1,0
MCI,2,2,2,2,0
EVE,3,3,3,4,1
NFO,4,4,4,3,1
NEW,5,5,5,5,0
MUN,6,6,6,6,0
TOT,10,7,7,8,3
AVL,7,8,10,10,3
CHE,9,9,9,9,0


In [18]:
# Short run vs long run - who to buy now, and who to wait for.
print("ATTACK - rank over different windows:")
display(compare_windows(outlook, windows=(3, WINDOW, LOOKAHEAD), metric="attack").head(12))
print("A team that improves as the window lengthens has a hard patch first -")
print("worth planning for rather than buying today.")

ATTACK - rank over different windows:


,next_3,next_5,next_10,swing
team,,,,
MCI,1,1,1,0
ARS,2,4,4,2
MUN,3,2,2,1
CHE,4,3,3,1
NEW,5,5,8,3
EVE,6,11,11,5
BRE,7,8,7,1
BHA,8,7,5,3
LIV,9,6,6,3


A team that improves as the window lengthens has a hard patch first -
worth planning for rather than buying today.


In [19]:
print("DEFENCE - rank over different windows:")
display(compare_windows(outlook, windows=(3, WINDOW, LOOKAHEAD), metric="defence").head(12))
print("Compare with the attack table above: a team can be a good short-term")
print("defensive pick and a poor attacking one, or the reverse.")

DEFENCE - rank over different windows:


,next_3,next_5,next_10,swing
team,,,,
ARS,1,1,1,0
MCI,2,2,2,0
EVE,3,3,4,1
NFO,4,4,3,1
NEW,5,5,6,1
MUN,6,6,5,1
TOT,7,7,10,3
LEE,8,11,8,3
BHA,9,15,16,7


Compare with the attack table above: a team can be a good short-term
defensive pick and a poor attacking one, or the reverse.


In [20]:
# Your own players' fixture runs
if squad_df is not None and len(squad_df):
    rk = rank_fixtures(outlook, window=WINDOW, discount=DISCOUNT, metric="overall")
    mine_fx = (squad_df[["web_name", "team", "position", "price", "xp"]]
               .merge(rk[["score", "xg_for", "clean_sheet", "blanks", "opponents"]],
                      left_on="team", right_index=True, how="left")
               .sort_values("score", ascending=False))
    print(f"Your squad by fixture run over the next {WINDOW} gameweeks:")
    display(mine_fx.round(3))
    print("Players at the bottom are the natural transfer candidates, even if")
    print("their expected points this week look fine.")

Your squad by fixture run over the next 5 gameweeks:


,web_name,team,position,price,xp,score,xg_for,clean_sheet,blanks,opponents
0,Haaland,MCI,FWD,15.5,7.834,2.121,1.965,0.337,0,"SUN(H), LIV(A), IPS(H), AVL(A), BHA(H)"
160,O'Reilly,MCI,DEF,6.4,2.539,2.121,1.965,0.337,0,"SUN(H), LIV(A), IPS(H), AVL(A), BHA(H)"
80,Raya,ARS,GKP,6.0,3.558,1.903,1.746,0.368,0,"BHA(A), LEE(H), NFO(A), EVE(H), LIV(A)"
117,Tzolis,ARS,MID,6.4,2.954,1.903,1.746,0.368,0,"BHA(A), LEE(H), NFO(A), EVE(H), LIV(A)"
33,Calafiori,ARS,DEF,5.8,4.378,1.903,1.746,0.368,0,"BHA(A), LEE(H), NFO(A), EVE(H), LIV(A)"
226,Shaw,MUN,DEF,4.4,1.734,0.850,1.804,0.248,0,"FUL(A), TOT(H), LEE(A), BOU(H), CHE(A)"
59,João Pedro,CHE,FWD,7.8,3.847,0.572,1.758,0.231,0,"BRE(A), BOU(H), EVE(A), TOT(H), MUN(H)"
10,Palmer,CHE,MID,9.7,5.400,0.572,1.758,0.231,0,"BRE(A), BOU(H), EVE(A), TOT(H), MUN(H)"
25,Szoboszlai,LIV,MID,7.0,4.668,-0.080,1.598,0.205,0,"BOU(A), MCI(H), BRE(A), BHA(H), ARS(H)"
207,Pedro Porro,TOT,DEF,5.5,2.037,-0.127,1.456,0.235,0,"AVL(H), MUN(A), COV(H), CHE(A), CRY(H)"


Players at the bottom are the natural transfer candidates, even if
their expected points this week look fine.


## 8. Multi-gameweek player ranking

Section 7 ranks *teams* by fixtures. This ranks *players* over the same
window, which is what a transfer decision actually needs - fixtures matter,
but so do minutes, role and price.

Each player's expected points are summed across the window with the same
`DISCOUNT` weighting.

### The assumptions, stated plainly

Minutes are frozen at today's estimate, except where FPL gives a return date.
Only about 13% of flagged players carry one ("Expected back 11 Oct"); those
switch back on at the right gameweek. The rest hold today's availability
across the whole window - deliberately, since we don't know when they return,
and a made-up recovery curve would look like information while being a guess.

A returning player's start rate comes from the price/position prior rather
than their own record, because their record is a run of zeros *caused by* the
injury and says nothing about whether they'd be picked when fit - several
have played no minutes at all this season. Those rows are tagged
`minutes_basis = "prior"` and flagged below; treat them as the roughest
numbers in the table.

Market odds only reach two to four gameweeks, and `market_share` reports how
much of each player's total came from priced fixtures rather than the ratings
model.

Prices are also frozen, even though over a long window they move and that
affects what you can afford later. Not modelled here.

In [21]:
from fplfh.availability import return_gameweeks
from fplfh.horizon import (player_horizon, gameweek_matrix,
                           suggest_transfers, transfer_summary)

return_gw = return_gameweeks(res.minutes, client.bootstrap()["events"])
print(f"{len(return_gw)} flagged players have a stated return gameweek")

HZN, per_gw = player_horizon(client, res.minutes, res.fixtures, res.event,
                             n_events=WINDOW, discount=DISCOUNT,
                             return_gw=return_gw, verbose=True)
print()
print(f"ranked {len(HZN)} players over GW{res.event}-{res.event + WINDOW - 1}, "
      f"discount {DISCOUNT}")
HZN.head(25)[["web_name", "team", "position", "price", "xp_total", "xp_raw",
              "xp_next", "fixtures", "blanks", "market_share",
              "xp_per_million"]].round(2)

26 flagged players have a stated return gameweek


odds covered 20/50 fixtures in GW5-9



ranked 659 players over GW5-9, discount 0.85


,web_name,team,position,price,xp_total,xp_raw,xp_next,fixtures,blanks,market_share,xp_per_million
0,Saka,ARS,MID,9.5,23.60,31.20,6.59,5,0,0.46,2.48
1,B.Fernandes,MUN,MID,12.0,23.18,31.02,6.27,5,0,0.43,1.93
2,Haaland,MCI,FWD,15.5,22.83,29.96,7.83,5,0,0.45,1.47
3,Barry,EVE,FWD,5.6,20.66,27.26,6.45,5,0,0.43,3.69
4,Mbeumo,MUN,MID,7.9,20.65,27.65,5.59,5,0,0.43,2.61
5,Gibbs-White,NFO,MID,8.0,20.62,27.31,6.62,5,0,0.43,2.58
6,Palmer,CHE,MID,9.7,19.54,26.25,5.20,5,0,0.42,2.01
7,Tavernier,BOU,MID,6.1,19.13,25.86,5.16,5,0,0.39,3.14
8,Rogers,CHE,MID,7.7,19.04,25.59,5.06,5,0,0.42,2.47
9,Isak,LIV,FWD,9.1,18.81,25.24,5.59,5,0,0.41,2.07


In [22]:
# Best value per million over the window, rather than for one gameweek
print("Best value per million over the window (xp_total > 5):")
display(HZN[HZN.xp_total > 5].nlargest(15, "xp_per_million")[
    ["web_name", "team", "position", "price", "xp_total", "xp_per_million",
     "blanks"]].round(2))

Best value per million over the window (xp_total > 5):


,web_name,team,position,price,xp_total,xp_per_million,blanks
30,Thomas,COV,DEF,4.0,15.37,3.84,0
3,Barry,EVE,FWD,5.6,20.66,3.69,0
16,Rudoni,COV,MID,5.0,17.20,3.44,0
49,Egan,HUL,DEF,4.1,14.05,3.43,0
69,Davis,IPS,DEF,4.0,13.23,3.31,0
43,Bogle,LEE,DEF,4.5,14.65,3.26,0
11,E.Le Fée,SUN,MID,5.8,18.65,3.22,0
17,Hall,NEW,DEF,5.2,16.65,3.20,0
22,Khalaili,CRY,DEF,5.0,15.98,3.20,0
68,Ajayi,HUL,DEF,4.2,13.25,3.16,0


In [23]:
# Who is riding a prior rather than a record? These are the shakiest rows.
shaky = HZN[HZN.get("minutes_basis", "observed") != "observed"]
if len(shaky):
    print("Returning from injury - minutes rest on the price/position prior,")
    print("not on anything observed. Sanity-check these by eye:")
    display(shaky.nlargest(10, "xp_total")[
        ["web_name", "team", "position", "price", "xp_total", "xmins_mean"]].round(2))
else:
    print("No players in the window are running on a prior.")

Returning from injury - minutes rest on the price/position prior,
not on anything observed. Sanity-check these by eye:


,web_name,team,position,price,xp_total,xmins_mean
197,Hinshelwood,BHA,MID,5.9,7.99,32.37
228,Palestra,CHE,DEF,5.3,6.26,45.37
232,Doku,MCI,MID,7.4,6.15,35.41
234,Goretzka,AVL,MID,5.9,5.97,34.76
245,Mitoma,BHA,MID,5.9,5.21,31.99
246,Foden,MCI,MID,7.0,5.15,23.39
250,Burn,NEW,DEF,4.9,5.05,34.99
253,Jensen,BRE,MID,5.4,4.94,30.92
256,Mateta,CRY,FWD,6.3,4.83,24.66
263,Reinildo,SUN,DEF,4.5,4.67,35.03


In [24]:
# The shape of a run matters as much as the total - a flat 5 a week is worth
# more to plan around than a 12 followed by four blanks.
print("Expected points by gameweek, top 8 over the window:")
gameweek_matrix(per_gw, HZN.head(8).web_name.tolist()).round(2)

Expected points by gameweek, top 8 over the window:


event,5,6,7,8,9
web_name,,,,,
B.Fernandes,6.27,6.95,5.48,6.57,5.74
Barry,6.45,5.34,5.98,4.23,5.26
Gibbs-White,6.62,5.03,5.28,5.33,5.05
Haaland,7.83,5.60,5.92,4.59,6.01
Mbeumo,5.59,6.17,4.91,5.85,5.13
Palmer,5.43,6.21,4.66,5.82,5.38
Saka,6.59,7.81,5.05,6.42,5.32
Tavernier,5.16,4.89,5.62,4.67,5.52


## 9. Transfers

For each player you own, the best affordable replacement over the window.

This is a deliberately simple greedy view: it takes the single biggest
upgrade, strikes both players off, then finds the next best among what's
left - tracking the bank and the three-per-club limit as it goes, so the list
is actually executable in order rather than just a set of individually
plausible swaps.

It doesn't consider combinations (selling two cheap players to fund one
expensive one), plan across future gameweeks, or model free-transfer banking.
Those would be a much larger optimisation, worth building only once the
projections have been validated enough to trust at that resolution.

Read `net` carefully: the gain is a whole-window total, while the −4 hit is
paid once. And a net under about a point is well inside the model's own
error - the backtest puts rank correlation around 0.69, useful for ordering
players but not for splitting hairs.

In [25]:
if squad_df is not None and len(squad_df):
    my_ids = squad_df.player_id.tolist()
    my_bank = 0.0
    if team_id:
        my_bank = client.latest_picks(int(team_id))[0]["entry_history"]["bank"] / 10.0

    owned = HZN[HZN.player_id.isin(my_ids)].sort_values("xp_total", ascending=False)
    print(f"Your squad over GW{res.event}-{res.event + WINDOW - 1} "
          f"(bank L{my_bank:.1f}m):")
    display(owned[["web_name", "team", "position", "price", "xp_total",
                   "xp_next", "blanks", "market_share"]].round(2))
    print(f"squad total (discounted): {owned.xp_total.sum():.1f}")
else:
    print("No team set - see section 1.")

Your squad over GW5-9 (bank L1.3m):


,web_name,team,position,price,xp_total,xp_next,blanks,market_share
2,Haaland,MCI,FWD,15.5,22.83,7.83,0,0.45
6,Palmer,CHE,MID,9.7,19.54,5.20,0,0.42
14,Calvert-Lewin,LEE,FWD,6.0,17.42,5.61,0,0.37
21,Szoboszlai,LIV,MID,7.0,16.05,4.67,0,0.40
25,Calafiori,ARS,DEF,5.8,15.88,4.38,0,0.47
41,Groß,BHA,MID,5.7,14.75,3.47,0,0.36
57,João Pedro,CHE,FWD,7.8,13.74,3.67,0,0.43
71,Raya,ARS,GKP,6.0,13.15,3.56,0,0.45
122,Tzolis,ARS,MID,6.4,10.75,2.95,0,0.44
132,Diop,IPS,DEF,4.0,10.46,2.59,0,0.40


squad total (discounted): 185.7


In [26]:
if squad_df is not None and len(squad_df):
    sug = suggest_transfers(HZN, my_ids, bank=my_bank)
    plan = transfer_summary(sug, bank=my_bank, squad_teams=owned.team.tolist(),
                            free_transfers=FREE_TRANSFERS)
    if len(plan):
        print(f"{len(sug)} legal upgrades found. Best executable sequence:")
        display(plan.head(10).round(2))
        print("bank_after tracks the money left once each swap is done, so the")
        print("sequence is affordable in the order shown.")
        good = plan[plan.net > 1.0]
        print()
        print(f"{len(good)} of these clear a net of +1.0, which is roughly the")
        print("threshold below which the model cannot tell the difference.")
    else:
        print("No affordable upgrades found - your squad is already strong for")
        print("this window, or the bank is too thin to move.")

37 legal upgrades found. Best executable sequence:


,transfer_no,out,out_team,out_price,out_xp,in,in_team,in_price,in_xp,gain,hit,net,cumulative_net,cost,bank_after,in_minutes_basis
0,1,Hughes,CRY,4.4,1.85,Rudoni,COV,5.0,17.20,15.36,0.0,15.36,15.36,0.6,0.7,observed
1,2,Pedro Porro,TOT,5.5,6.38,Hall,NEW,5.2,16.65,10.27,4.0,6.27,21.63,-0.3,1.0,observed
2,3,Shaw,MUN,4.4,6.21,Khalaili,CRY,5.0,15.98,9.77,4.0,5.77,27.40,0.6,0.4,observed
3,4,O'Reilly,MCI,6.4,6.97,Murillo,NFO,5.5,16.24,9.27,4.0,5.27,32.67,-0.9,1.3,observed
4,5,Tzolis,ARS,6.4,10.75,Tavernier,BOU,6.1,19.13,8.38,4.0,4.38,37.05,-0.3,1.6,observed
5,6,João Pedro,CHE,7.8,13.74,Barry,EVE,5.6,20.66,6.91,4.0,2.91,39.96,-2.2,3.8,observed
6,7,Diop,IPS,4.0,10.46,Thomas,COV,4.0,15.37,4.91,4.0,0.91,40.87,0.0,3.8,observed
7,8,Szoboszlai,LIV,7.0,16.05,Mbeumo,MUN,7.9,20.65,4.61,4.0,0.61,41.48,0.9,2.9,observed
8,9,Groß,BHA,5.7,14.75,E.Le Fée,SUN,5.8,18.65,3.91,4.0,-0.09,41.39,0.1,2.8,observed
9,10,Verbruggen,BHA,4.5,9.73,Pickford,EVE,5.5,12.33,2.60,4.0,-1.40,39.99,1.0,1.8,observed


bank_after tracks the money left once each swap is done, so the
sequence is affordable in the order shown.

6 of these clear a net of +1.0, which is roughly the
threshold below which the model cannot tell the difference.


## 10. Minutes worth checking by hand

The model reads FPL's availability flags but cannot hear a press conference.
These are the players where its guess matters most and is least certain - worth
overriding in `config/minutes_overrides.yaml` if you know better, then re-running.

In [27]:
m = res.minutes
risky = res.players.merge(
    m[["player_id","p_start","p_60","status","news"]], on="player_id", suffixes=("","_m"))
risky = risky[(risky.xp > 1.5) & (risky.p_start.between(0.2, 0.85))]
print("Rotation risks among players worth owning:\n")
display(risky.nlargest(15, "xp")[
    ["web_name","team","position","price","opponent","p_start","p_60","xmins","xp"]].round(3))

flagged = m[(m.status != "a") & (m.price >= 4.5)]
if len(flagged):
    print("\nFlagged players (automatically downgraded):\n")
    display(flagged.nlargest(12, "price")[
        ["web_name","team","position","price","status","chance_of_playing",
         "p_start","xmins","news"]])

Rotation risks among players worth owning:



,web_name,team,position,price,opponent,p_start,p_60,xmins,xp
32,Cherki,MCI,MID,7.8,SUN,0.761,0.534,58.329,4.391
41,Rudoni,COV,MID,5.0,NFO,0.817,0.797,70.652,4.134
45,Marmoush,TOT,FWD,7.0,AVL,0.797,0.778,68.834,4.010
51,Neto,CHE,MID,6.5,BRE,0.809,0.790,70.397,3.924
59,João Pedro,CHE,FWD,7.8,BRE,0.686,0.671,60.425,3.847
60,Van de Ven,TOT,DEF,5.0,AVL,0.822,0.803,70.784,3.846
61,Garner,EVE,MID,6.0,IPS,0.796,0.777,69.561,3.829
64,Mainoo,MUN,MID,5.5,FUL,0.785,0.766,68.259,3.776
76,Bentancur,TOT,MID,5.5,AVL,0.807,0.787,69.554,3.598
77,Elvedi,LEE,DEF,4.5,CRY,0.623,0.605,56.375,3.572



Flagged players (automatically downgraded):



,web_name,team,position,price,status,chance_of_playing,p_start,xmins,news
51,Watkins,AVL,FWD,7.8,u,0.0,0.000000,0.000000,Has joined Al Hilal permanently
181,João Pedro,CHE,FWD,7.8,d,75.0,0.685633,60.425194,Unspecified injury - 75% chance of playing
85,Kroupi.Jr,BOU,MID,7.4,i,0.0,0.000000,0.000000,Foot injury - Expected back 7 Nov
456,Ekitiké,LIV,FWD,7.4,i,0.0,0.000000,0.000000,Achilles injury - Unknown return date
479,Doku,MCI,MID,7.4,i,0.0,0.000000,0.000000,Calf injury - Expected back 20 Sep
477,Foden,MCI,MID,7.0,s,0.0,0.000000,0.000000,Suspended until 17 Oct
480,Rodrigo,MCI,MID,6.5,u,0.0,0.000000,0.000000,Has joined Barcelona permanently
619,Kulusevski,TOT,MID,6.5,i,0.0,0.000000,0.000000,Knee injury - Unknown return date
17,Martinelli,ARS,MID,6.3,u,0.0,0.000000,0.000000,Has joined Al Hilal permanently
251,Mateta,CRY,FWD,6.3,i,0.0,0.000000,0.000000,Hamstring injury - Expected back 11 Oct


## 11. Export

In [28]:
from fplfh.config import OUT_DIR, ensure_dirs
ensure_dirs()
tag = f"gw{res.event}"
ALL.to_csv(OUT_DIR / f"all_players_{tag}.csv", index=False)
res.fixture_table().to_csv(OUT_DIR / f"fixtures_{tag}.csv", index=False)
if squad_df is not None and len(squad_df):
    squad_df.to_csv(OUT_DIR / f"my_squad_{tag}.csv", index=False)
print("written to", OUT_DIR)

written to

 C:\Users\alexg\OneDrive\Desktop\Projects\Fantasy-PL\data\out
